# ==============================================================================
# Step 2: High-Performance Unified BN-Adaptation Sweep across All Models
# ==============================================================================
# Generates each corrupted batch ONCE and evaluates ALL candidate models concurrently on GPU.
# Eliminates redundant CPU corruptions (6x-9x speedup, 10-15 mins total).
# Generates evaluation_summary.json, robustness_matrix.csv, and preds_*.npy for every model.

!python scripts/evaluate_all_models_unified.py \
    --reports-dir reports \
    --batch-size 128 \
    --tau-shift 0.10


In [ ]:
# ==============================================================================
# Step 0: Environment Setup, Drive Mount & GPU Verification
# ==============================================================================
import os
import sys
from pathlib import Path

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Configure workspace
WORKSPACE_DIR = Path('/content/ML_Project')
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(str(WORKSPACE_DIR))

if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))
os.environ['PYTHONPATH'] = f"{WORKSPACE_DIR}:" + os.environ.get('PYTHONPATH', '')

# 3. Verify CUDA T4 GPU
import torch
print("=" * 75)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")
print("=" * 75)
!nvidia-smi

# 4. Extract Archive if uploaded to /content or Google Drive
archive_candidates = [
    Path('/content/drive/MyDrive/colab_code_package_with_weights.zip'),
    Path('/content/colab_code_package_with_weights.zip'),
    Path('/content/drive/MyDrive/colab_code_package.zip'),
    Path('/content/colab_code_package.zip'),
]
CODE_ARCHIVE = None
for cand in archive_candidates:
    if cand.exists():
        CODE_ARCHIVE = cand
        break

if CODE_ARCHIVE is not None:
    print(f"Unpacking {CODE_ARCHIVE} into {WORKSPACE_DIR}...")
    !unzip -q -o {CODE_ARCHIVE} -d {WORKSPACE_DIR}
else:
    print("Note: No zip archive detected in /content or Drive. Using existing workspace files.")

In [ ]:
# ==============================================================================
# Step 0B: Link Raw GTSRB Dataset
# ==============================================================================
# Link raw dataset from Google Drive or extract archive
DRIVE_DATA = Path('/content/drive/MyDrive/german')
LOCAL_DATA = WORKSPACE_DIR / 'german'

if not LOCAL_DATA.exists():
    if DRIVE_DATA.exists():
        print(f"Creating symlink from {DRIVE_DATA} to {LOCAL_DATA}...")
        !ln -s {DRIVE_DATA} {LOCAL_DATA}
    else:
        DATA_ZIP = Path('/content/drive/MyDrive/ML_Project_Data.zip')
        if DATA_ZIP.exists():
            print(f"Extracting {DATA_ZIP}...")
            !unzip -q -o {DATA_ZIP} -d /content/
            if Path('/content/german').exists() and not LOCAL_DATA.exists():
                !ln -s /content/german {LOCAL_DATA}

assert LOCAL_DATA.exists(), f"Error: Raw GTSRB data not found at {LOCAL_DATA}! Please check Drive mount."
print(f"GTSRB Data Verified at: {LOCAL_DATA}")

In [ ]:
# ==============================================================================
# Step 1: Self-Healing Checkpoint Fallback & Automated Model Training
# ==============================================================================
# Checks if checkpoints exist; if missing, automatically trains them with seed 42.

checkpoints_config = {
    "exp1_baseline": {
        "path": Path("reports/baseline_resnet18/best_baseline_resnet18.pth"),
        "drive_alt": Path("/content/drive/MyDrive/checkpoints/baseline_resnet18/best_baseline_resnet18.pt"),
        "train_cmd": "python train_baseline.py --epochs 25 --output-dir reports/baseline_resnet18",
    },
    "exp2_augmix": {
        "path": Path("reports/augmix_resnet18/best_augmix_resnet18.pth"),
        "drive_alt": Path("/content/drive/MyDrive/checkpoints/augmix_resnet18/best_augmix_resnet18.pt"),
        "train_cmd": "python train_augmix.py --epochs 25 --output-dir reports/augmix_resnet18",
    },
    "exp3_frequency": {
        "path": Path("reports/frequency_exp3/best_frequency_resnet18.pt"),
        "drive_alt": Path("/content/drive/MyDrive/checkpoints/frequency_resnet18/best_frequency_resnet18.pt"),
        "train_cmd": "python train_frequency.py --epochs 25 --output-dir reports/frequency_exp3",
    },
    "exp4_cbm": {
        "path": Path("reports/cbm_resnet18/best_cbm_resnet18.pt"),
        "drive_alt": Path("/content/drive/MyDrive/checkpoints/cbm_resnet18/best_cbm_resnet18.pt"),
        "train_cmd": "python train_cbm.py --epochs 25 --output-dir reports/cbm_resnet18",
    },
    "exp5_hybrid": {
        "path": Path("reports/hybrid_resnet18/best_hybrid_resnet18.pt"),
        "drive_alt": Path("/content/drive/MyDrive/checkpoints/hybrid_resnet18/best_hybrid_resnet18.pt"),
        "train_cmd": "python train_hybrid.py --epochs 25 --ablation none --output-dir reports/hybrid_resnet18",
    },
    "exp6_freq_cbm": {
        "path": Path("reports/freq_cbm_resnet18/best_freq_cbm_resnet18.pt"),
        "drive_alt": Path("/content/drive/MyDrive/checkpoints/freq_cbm_resnet18/best_freq_cbm_resnet18.pt"),
        "train_cmd": "python train_freq_cbm.py --epochs 25 --output-dir reports/freq_cbm_resnet18",
    },
    "ablation_5a_geom": {
        "path": Path("reports/hybrid_resnet18/best_hybrid_geometric_only_resnet18.pt"),
        "drive_alt": Path("/content/drive/MyDrive/checkpoints/hybrid_resnet18/best_hybrid_geometric_only_resnet18.pt"),
        "train_cmd": "python train_hybrid.py --epochs 25 --ablation geometric_only --output-dir reports/hybrid_resnet18",
    },
    "ablation_5b_decouple": {
        "path": Path("reports/hybrid_resnet18/best_hybrid_decouple_concept_jsd_resnet18.pt"),
        "drive_alt": Path("/content/drive/MyDrive/checkpoints/hybrid_resnet18/best_hybrid_decouple_concept_jsd_resnet18.pt"),
        "train_cmd": "python train_hybrid.py --epochs 25 --ablation decouple_concept_jsd --output-dir reports/hybrid_resnet18",
    },
    "exp8_foveated_cbm": {
        "path": Path("reports/foveated_cbm_resnet18/best_foveated_cbm_resnet18.pt"),
        "drive_alt": Path("/content/drive/MyDrive/checkpoints/foveated_cbm_resnet18/best_foveated_cbm_resnet18.pt"),
        "train_cmd": "python train_foveated_cbm.py --epochs 25 --output-dir reports/foveated_cbm_resnet18",
    },
}

for key, cfg in checkpoints_config.items():
    target_p = cfg["path"]
    drive_p = cfg["drive_alt"]
    
    # Check if alternate extension (.pt vs .pth) exists locally
    if not target_p.exists():
        alt_p = target_p.with_suffix('.pt' if target_p.suffix == '.pth' else '.pth')
        if alt_p.exists():
            target_p = alt_p
            cfg["path"] = alt_p
    
    # Copy from Drive if present
    if not target_p.exists() and drive_p.exists():
        print(f"[{key}] Restoring checkpoint from Google Drive: {drive_p} -> {target_p}")
        target_p.parent.mkdir(parents=True, exist_ok=True)
        !cp {drive_p} {target_p}
        
    # Self-healing fallback: train if missing
    if not target_p.exists():
        print(f"[{key}] Checkpoint NOT found at {target_p}! Triggering automated self-healing training...")
        print(f"  Command: {cfg['train_cmd']}")
        !{cfg['train_cmd']}
    else:
        print(f"[{key}] Checkpoint Verified: {target_p} ({target_p.stat().st_size / (1024*1024):.1f} MB)")

In [ ]:
# ==============================================================================
# Step 2: High-Performance Unified BN-Adaptation Sweep across All Models
# ==============================================================================
# Generates each corrupted batch ONCE and evaluates ALL candidate models concurrently on GPU.
# Eliminates redundant CPU corruptions (6x-9x speedup, 10-15 mins total).
# Generates evaluation_summary.json, robustness_matrix.csv, and preds_*.npy for every model.

!python scripts/evaluate_all_models_unified.py \
    --reports-dir reports \
    --batch-size 128 \
    --tau-shift 0.10


In [ ]:
# ==============================================================================
# Step 3: Test-Time Counterfactual Concept Intervention Benchmark (intervene())
# ==============================================================================
# Evaluates Modes 0 to 6 across Severities 3, 4, 5 under Static and Shift-Gated BN
# Quantifies Cross-Group vs Within-Group errors and post-adaptation CDR

cbm_models = [
    ("cbm_exp4", "reports/cbm_resnet18/best_cbm_resnet18.pt", "reports/diagnostics/concept_intervention_exp4.json"),
    ("freq_cbm_exp6", "reports/freq_cbm_resnet18/best_freq_cbm_resnet18.pt", "reports/diagnostics/concept_intervention_exp6.json"),
]

for tag, ckpt, out_json in cbm_models:
    if Path(ckpt).exists():
        print("\n" + "=" * 80)
        print(f"RUNNING CONCEPT INTERVENTION BENCHMARK: {tag} ({ckpt})")
        print("=" * 80)
        !python scripts/evaluate_concept_intervention.py \
            --checkpoint {ckpt} \
            --output {out_json} \
            --batch-size 128 \
            --tau-shift 0.10
    else:
        print(f"Skipping {tag}; checkpoint {ckpt} not found.")

In [ ]:
# ==============================================================================
# Step 4: Temporal Video-Track Inference & Concept Smoothing (30-Frame Sequences)
# ==============================================================================
# Evaluates on val.csv (261 tracks, 30 frames per track = 7,830 images)
# Compares Single-Frame vs Temporal EMA (gamma=0.5, 0.7, 0.85) vs Online BN-Adapt (N=8)

TEMPORAL_CKPT = "reports/cbm_resnet18/best_cbm_resnet18.pt"
if Path(TEMPORAL_CKPT).exists():
    print("\n" + "=" * 80)
    print(f"RUNNING TEMPORAL VIDEO-TRACK BENCHMARK (All 261 Tracks)")
    print("=" * 80)
    !python scripts/evaluate_temporal_tracks.py \
        --checkpoint {TEMPORAL_CKPT} \
        --val-meta data/master_v1/metadata/val.csv \
        --output reports/diagnostics/temporal_track_smoothing.json
else:
    print(f"Skipping temporal evaluation; {TEMPORAL_CKPT} not found.")

In [ ]:
# ==============================================================================
# Step 5: Automated Pairwise Statistical Significance & 1,000-Iteration Bootstrap
# ==============================================================================
# Loads generated preds_static.npy and preds_bn_adapt.npy from all models
# Computes McNemar's chi^2, p-values, and 95% Bootstrap Confidence Intervals

print("\n" + "=" * 80)
print("RUNNING PAIRWISE STATISTICAL SIGNIFICANCE & BOOTSTRAP SUITE")
print("=" * 80)

!python scripts/compute_statistical_significance.py \
    --reports-dir reports \
    --test-meta data/master_v1/metadata/test.csv \
    --output reports/diagnostics/statistical_significance_report.json \
    --n-bootstrap 1000

In [ ]:
# ==============================================================================
# Step 6: Direct Sync to Google Drive (MyDrive/ML_Project_Results) & Zip Download
# ==============================================================================
import os
import shutil
import zipfile
from pathlib import Path
from google.colab import files

# 1. Package all evaluation files and prediction arrays into zip
RESULTS_ZIP = Path('/content/final_experimental_results.zip')
print(f"Packaging all generated reports and prediction arrays into {RESULTS_ZIP}...")

include_patterns = [
    "reports/**/*.json",
    "reports/**/*.csv",
    "reports/**/*.npy",
]

with zipfile.ZipFile(RESULTS_ZIP, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for pattern in include_patterns:
        for p in WORKSPACE_DIR.glob(pattern):
            rel_p = p.relative_to(WORKSPACE_DIR)
            zipf.write(p, arcname=str(rel_p))
            print(f"  Packed: {rel_p}")

zip_size_mb = RESULTS_ZIP.stat().st_size / (1024 * 1024)
print(f"\nZip created successfully! Total size: {zip_size_mb:.2f} MB")

# 2. Direct Sync to Google Drive: MyDrive/ML_Project_Results (Capital R)
DRIVE_RESULTS = Path('/content/drive/MyDrive/ML_Project_Results')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print(f"Syncing all reports, weights, and prediction matrices to {DRIVE_RESULTS}...")

shutil.copytree(WORKSPACE_DIR / 'reports', DRIVE_RESULTS / 'reports', dirs_exist_ok=True)
shutil.copy2(RESULTS_ZIP, DRIVE_RESULTS / RESULTS_ZIP.name)
print("\n" + "=" * 80)
print(f"VERIFIED: All weights, matrices, and reports saved to: {DRIVE_RESULTS}")
print("=" * 80)

# 3. Trigger Browser Download
print("Triggering browser download...")
files.download(str(RESULTS_ZIP))
